# CSC-44112 Assessment Part 2
# Customer Response Prediction Using Machine Learning

This notebook is written in simple sections so it can be matched clearly to the report.
Run the notebooks in order from 01 to 04.

## Notebook 2: Preprocessing and Feature Engineering
This notebook prepares the data for machine learning.

In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
import joblib

RAW_PATH = Path('../data/raw/marketing_campaign.csv')
PROCESSED_DIR = Path('../data/processed')
MODEL_DIR = Path('../models')
PROCESSED_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)

df = pd.read_csv(RAW_PATH, sep='\t')
df.head()

### Cleaning and feature engineering

In [ ]:
df = df.dropna().copy()
df['Dt_Customer'] = pd.to_datetime(df['Dt_Customer'], errors='coerce')
df['Age'] = 2025 - df['Year_Birth']
df['Total_Spending'] = (
    df['MntWines'] + df['MntFruits'] + df['MntMeatProducts'] +
    df['MntFishProducts'] + df['MntSweetProducts'] + df['MntGoldProds']
)

# Remove ID because it does not help prediction.
df = df.drop(columns=['ID'])

df.head()

### Encode categorical variables
Education and Marital_Status are converted into numbers.

In [ ]:
label_encoders = {}
for col in ['Education', 'Marital_Status']:
    encoder = LabelEncoder()
    df[col] = encoder.fit_transform(df[col])
    label_encoders[col] = encoder

joblib.dump(label_encoders, MODEL_DIR / 'label_encoders.pkl')
df[['Education', 'Marital_Status']].head()

### Create feature matrix X and target y

In [ ]:
X = df.drop(columns=['Response', 'Dt_Customer'])
y = df['Response']

print('X shape:', X.shape)
print('y shape:', y.shape)

### Train-test split
An 80/20 split is used. Stratify keeps the class distribution similar in training and test data.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Training rows:', X_train.shape[0])
print('Testing rows:', X_test.shape[0])

### Feature scaling
Scaling is mainly needed for Logistic Regression.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

joblib.dump(scaler, MODEL_DIR / 'scaler.pkl')

### Save processed files

In [ ]:
pd.DataFrame(X_train).to_csv(PROCESSED_DIR / 'X_train.csv', index=False)
pd.DataFrame(X_test).to_csv(PROCESSED_DIR / 'X_test.csv', index=False)
pd.Series(y_train).to_csv(PROCESSED_DIR / 'y_train.csv', index=False)
pd.Series(y_test).to_csv(PROCESSED_DIR / 'y_test.csv', index=False)

pd.DataFrame(X_train_scaled, columns=X_train.columns).to_csv(PROCESSED_DIR / 'X_train_scaled.csv', index=False)
pd.DataFrame(X_test_scaled, columns=X_test.columns).to_csv(PROCESSED_DIR / 'X_test_scaled.csv', index=False)

print('Processed datasets saved.')